In [ ]:
%%capture
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install pennylane
!pip install torchinfo

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import timedelta

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms, models

from sklearn.metrics import *
from tqdm.auto import tqdm

import pennylane as qml
from torchinfo import summary

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 6
EPOCHS     = 25
LR         = 0.00005
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
CSV_PATH = "/kaggle/input/datasets/farjanakabirsamanta/skin-cancer-dataset/HAM10000_metadata.csv"
IMG_DIR  = "/kaggle/input/datasets/farjanakabirsamanta/skin-cancer-dataset/Skin Cancer/Skin Cancer"

## Augmentation Strategy

**Key principle: split on original image IDs first, augment training set only.**

Classes (alphabetical, matching `sorted(df['dx'].unique())`):
- **Minority** (akiec, bcc, df, vasc) → **4 sets** in training: original + rotation + zoom + brightness  
- **Majority** (bkl, mel, nv)         → **2 sets** in training: original + rotation  
- **Val & Test**: original images only, no augmentation (no leakage)

## Step 1 — Split image IDs (before any augmentation)

In [ ]:
df_meta = pd.read_csv(CSV_PATH)

# Keep only images that actually exist on disk
df_meta['path'] = df_meta['image_id'].apply(
    lambda x: os.path.join(IMG_DIR, x + '.jpg')
)
df_meta = df_meta[df_meta['path'].apply(os.path.exists)].reset_index(drop=True)

# ── Global label mapping ────────────────────────────────────────────────────────
MINORITY_CLASSES = ['akiec', 'bcc', 'df', 'vasc']
MAJORITY_CLASSES = ['bkl', 'mel', 'nv']
ALL_CLASSES      = sorted(MINORITY_CLASSES + MAJORITY_CLASSES)
CLASS_TO_IDX     = {cls: i for i, cls in enumerate(ALL_CLASSES)}
CLASSES          = list(CLASS_TO_IDX.keys())
num_classes      = len(CLASSES)

df_meta['label'] = df_meta['dx'].map(CLASS_TO_IDX)

print("Class → Index:")
for cls, idx in CLASS_TO_IDX.items():
    role = 'MINORITY' if cls in MINORITY_CLASSES else 'MAJORITY'
    print(f"  {idx}: {cls:8s} [{role}]")

print(f"\nTotal valid images: {len(df_meta):,}")

In [ ]:
# ── 70 / 15 / 15 split on image IDs — stratified by class ─────────────────────
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_meta, test_size=0.30, random_state=42, stratify=df_meta['dx']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['dx']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Original split sizes — train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")
print("\nNo image_id appears in more than one split:")
assert len(set(train_df.image_id) & set(val_df.image_id))  == 0
assert len(set(train_df.image_id) & set(test_df.image_id)) == 0
assert len(set(val_df.image_id)   & set(test_df.image_id)) == 0
print("  ✓ All clear — zero overlap between splits")

## Step 2 — Define Transforms

In [ ]:
# Used for val, test, and the 'original' training set
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# For majority training augmentation (rotation only)
rotation_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=30),
    transforms.ToTensor(),
])

# Minority augmentations
zoom_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(size=IMG_SIZE, scale=(0.7, 1.0)),
    transforms.ToTensor(),
])

brightness_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.5, contrast=0.3),
    transforms.ToTensor(),
])

print("Transforms defined:")
print("  base_transform       : Resize → ToTensor  (val/test + original train)")
print("  rotation_transform   : Resize → RandomRotation(±30°) → ToTensor")
print("  zoom_transform       : Resize → RandomResizedCrop(0.7–1.0) → ToTensor")
print("  brightness_transform : Resize → ColorJitter(brightness±0.5) → ToTensor")

## Step 3 — Dataset Class (takes a pre-split DataFrame)

In [ ]:
class SkinCancerDataset(Dataset):
    """
    Loads images from a pre-split DataFrame row set.
    Optionally filters to a subset of classes.
    """
    def __init__(self, dataframe, transform=None, filter_classes=None):
        """
        Args:
            dataframe      : DataFrame slice (train_df / val_df / test_df).
                             Must have columns: 'path', 'label', 'dx'.
            transform      : torchvision transform to apply.
            filter_classes : list of class names to keep (None = keep all).
        """
        if filter_classes is not None:
            dataframe = dataframe[dataframe['dx'].isin(filter_classes)]

        self.image_paths = dataframe['path'].tolist()
        self.labels      = dataframe['label'].tolist()
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

## Step 4 — Visualise Augmentations (before training)

In [ ]:
def show_augmentation_grid(train_dataframe, minority_classes, majority_classes,
                           n_examples=3):
    """
    Shows side-by-side augmentation previews using only training images.
    Minority: 4 columns (original, rotation, zoom, brightness)
    Majority: 2 columns (original, rotation)
    """
    min_transforms = [
        ('Original',   base_transform),
        ('Rotation',   rotation_transform),
        ('Zoom',       zoom_transform),
        ('Brightness', brightness_transform),
    ]
    maj_transforms = [
        ('Original', base_transform),
        ('Rotation', rotation_transform),
    ]

    def t2img(tensor):
        return np.clip(tensor.permute(1, 2, 0).numpy(), 0, 1)

    # Sample from training set only
    min_rows = train_dataframe[train_dataframe['dx'].isin(minority_classes)].sample(
        n_examples, random_state=7)
    maj_rows = train_dataframe[train_dataframe['dx'].isin(majority_classes)].sample(
        n_examples, random_state=7)

    # ── Minority grid ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(n_examples, 4, figsize=(16, 4 * n_examples),
                             constrained_layout=True)
    fig.suptitle(
        'MINORITY CLASS AUGMENTATIONS  (akiec / bcc / df / vasc)\n'
        '4 sets: original · rotation · zoom · brightness',
        fontsize=13, fontweight='bold', y=1.02
    )
    for r, (_, row) in enumerate(min_rows.iterrows()):
        pil = Image.open(row['path']).convert('RGB')
        for c, (lbl, tfm) in enumerate(min_transforms):
            ax = axes[r, c] if n_examples > 1 else axes[c]
            ax.imshow(t2img(tfm(pil)))
            ax.set_title(f'{lbl}\n[{row["dx"]}]', fontsize=9)
            ax.axis('off')
    plt.savefig('/kaggle/working/minority_augmentations.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Minority grid saved → /kaggle/working/minority_augmentations.png")

    # ── Majority grid ──────────────────────────────────────────────────────────
    fig2, axes2 = plt.subplots(n_examples, 2, figsize=(8, 4 * n_examples),
                               constrained_layout=True)
    fig2.suptitle(
        'MAJORITY CLASS AUGMENTATIONS  (bkl / mel / nv)\n'
        '2 sets: original · rotation',
        fontsize=13, fontweight='bold', y=1.02
    )
    for r, (_, row) in enumerate(maj_rows.iterrows()):
        pil = Image.open(row['path']).convert('RGB')
        for c, (lbl, tfm) in enumerate(maj_transforms):
            ax = axes2[r, c] if n_examples > 1 else axes2[c]
            ax.imshow(t2img(tfm(pil)))
            ax.set_title(f'{lbl}\n[{row["dx"]}]', fontsize=9)
            ax.axis('off')
    plt.savefig('/kaggle/working/majority_augmentations.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Majority grid saved → /kaggle/working/majority_augmentations.png")


show_augmentation_grid(train_df, MINORITY_CLASSES, MAJORITY_CLASSES, n_examples=3)

## Step 5 — Build Train / Val / Test Loaders

**Training set**: augmented copies concatenated  
**Val & Test**: original images only, no augmentation

In [ ]:
# ── TRAINING: augmented sets built from train_df only ─────────────────────────

# Minority → 4 sets
train_min_original   = SkinCancerDataset(train_df, base_transform,       MINORITY_CLASSES)
train_min_rotation   = SkinCancerDataset(train_df, rotation_transform,   MINORITY_CLASSES)
train_min_zoom       = SkinCancerDataset(train_df, zoom_transform,        MINORITY_CLASSES)
train_min_brightness = SkinCancerDataset(train_df, brightness_transform,  MINORITY_CLASSES)

# Majority → 2 sets
train_maj_original   = SkinCancerDataset(train_df, base_transform,       MAJORITY_CLASSES)
train_maj_rotation   = SkinCancerDataset(train_df, rotation_transform,   MAJORITY_CLASSES)

# Combine all training sets
train_dataset = ConcatDataset([
    train_min_original,
    train_min_rotation,
    train_min_zoom,
    train_min_brightness,
    train_maj_original,
    train_maj_rotation,
])

# ── VAL & TEST: base transform only, all classes, no augmentation ─────────────
val_dataset  = SkinCancerDataset(val_df,  base_transform)
test_dataset = SkinCancerDataset(test_df, base_transform)

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print("Dataset sizes:")
print(f"  Minority original training images : {len(train_min_original):,}")
print(f"  Minority × 4 (after augmentation) : {4 * len(train_min_original):,}")
print(f"  Majority original training images : {len(train_maj_original):,}")
print(f"  Majority × 2 (after augmentation) : {2 * len(train_maj_original):,}")
print(f"  Total training images             : {len(train_dataset):,}")
print(f"  Val images  (no augmentation)     : {len(val_dataset):,}")
print(f"  Test images (no augmentation)     : {len(test_dataset):,}")

## Quantum Circuit

In [ ]:
n_qubits = 2
n_layers = 2

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.RX(weights[l, i], wires=i)
            qml.RZ(weights[l, i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[n_qubits-1, 0])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(n_layers, n_qubits))

    def forward(self, x):
        outputs = []
        for i in range(x.shape[0]):
            out = quantum_circuit(x[i], self.weights)
            outputs.append(torch.stack(out).float())
        return torch.stack(outputs)

## Hybrid MobileNet Model

In [ ]:
class HybridMobileNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        base = models.mobilenet_v2(weights="DEFAULT")
        self.features = base.features

        freeze_layers = len(self.features) // 2
        for i in range(freeze_layers):
            for p in self.features[i].parameters():
                p.requires_grad = False

        self.q_branch = nn.Sequential(
            nn.Conv2d(160, 40, 1),
            nn.BatchNorm2d(40),
            nn.ReLU(),
            nn.Conv2d(40, 16, 3, padding=1, dilation=2),
            nn.BatchNorm2d(16),
            nn.SiLU(),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.q_fc    = nn.Linear(16, n_qubits)
        self.quantum = QuantumLayer()

        self.extra = nn.Sequential(
            nn.Conv2d(1280, 128, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.final_fc = nn.Linear(64 + n_qubits, num_classes)

    def forward(self, x):
        x = self.features[:17](x)

        q = self.q_branch(x)
        q = torch.flatten(q, 1)
        q = torch.tanh(self.q_fc(q))
        q = self.quantum(q)

        c = self.features[17:](x)
        c = self.extra(c)
        c = nn.functional.adaptive_avg_pool2d(c, (1, 1))
        c = torch.flatten(c, 1)

        return self.final_fc(torch.cat([c, q], dim=1))

In [ ]:
model     = HybridMobileNet(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

## Training Loop

In [ ]:
def custom_metrics(y_pred, y_true, loss):
    y_pred = torch.argmax(y_pred, 1)
    y_true = y_true.cpu().numpy()
    y_pred = y_pred.cpu().numpy()
    return {
        "loss":     round(loss, 3),
        "accuracy": round(accuracy_score(y_true, y_pred), 3)
    }

In [ ]:
train_hist = {"loss": [], "accuracy": []}
val_hist   = {"loss": [], "accuracy": []}

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    model.train()
    train_loss = 0
    preds, labels = [], []

    for x, y in tqdm(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds.append(out)
        labels.append(y)

    preds  = torch.cat(preds)
    labels = torch.cat(labels)
    train_metrics = custom_metrics(preds, labels, train_loss / len(train_loader))
    train_hist["loss"].append(train_metrics["loss"])
    train_hist["accuracy"].append(train_metrics["accuracy"])
    print("Train:", {k: f"{v:.3f}" for k, v in train_metrics.items()})

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0
    preds, labels = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out  = model(x)
            loss = criterion(out, y)
            val_loss += loss.item()
            preds.append(out)
            labels.append(y)

    preds  = torch.cat(preds)
    labels = torch.cat(labels)
    val_metrics = custom_metrics(preds, labels, val_loss / len(val_loader))
    val_hist["loss"].append(val_metrics["loss"])
    val_hist["accuracy"].append(val_metrics["accuracy"])
    print("Val:",   {k: f"{v:.3f}" for k, v in val_metrics.items()})

    if epoch + 1 == 15:
        torch.save(model.state_dict(), "/kaggle/working/hybrid_model_epoch_15.pt")
        print("✓ Model saved at epoch 15")
    elif epoch + 1 == 25:
        torch.save(model.state_dict(), "/kaggle/working/hybrid_model_epoch_25_final.pt")
        print("✓ Model saved at epoch 25 (final)")

## Training Curves

In [ ]:
epochs = range(1, len(train_hist["loss"]) + 1)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_hist["loss"],     label="Train")
plt.plot(epochs, val_hist["loss"],       label="Val")
plt.title("Loss"); plt.legend(); plt.grid()

plt.subplot(1, 2, 2)
plt.plot(epochs, train_hist["accuracy"], label="Train")
plt.plot(epochs, val_hist["accuracy"],   label="Val")
plt.title("Accuracy"); plt.legend(); plt.grid()

plt.tight_layout()
plt.show()

## Test Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out  = model(x)
        all_preds.append(out)
        all_labels.append(y)

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
pred_classes = torch.argmax(all_preds, dim=1)

report = classification_report(
    all_labels.cpu(), pred_classes.cpu(),
    target_names=CLASSES, digits=3
)
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(report)

In [ ]:
cm   = confusion_matrix(all_labels.cpu(), pred_classes.cpu())
disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()

## Parameter Summary

In [ ]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print("\n" + "="*40)
print(f"{'Total Parameters:':<25} {total_params:,}")
print(f"{'Trainable Parameters:':<25} {trainable_params:,}")
print(f"{'Frozen Parameters:':<25} {frozen_params:,}")
print("="*40)